# Doc-RAG: Full Backend Implementation
This notebook contains all the core logic from the FastAPI backend, consolidated for research and prototyping.

### 1. Configuration & Core Settings

In [ ]:
import os
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    PROJECT_NAME: str = "DocChat"
    PINECONE_INDEX_NAME: str = "doc-rag"
    PINECONE_DIMENSION: int = 768
    PINECONE_METRIC: str = "dotproduct"
    JWT_SECRET: str = "88d6b113-b342-4358-9c87-82344f77f7dd"
    
    class Config:
        env_file = ".env"

settings = Settings()

### 2. Vector Store Service (Hybrid Search)

In [ ]:
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(settings.PINECONE_INDEX_NAME)
bm25 = BM25Encoder().default()

def query_vectors(query_vector, top_k=5, namespace="default", sparse_vector=None):
    kwargs = {
        "vector": query_vector,
        "top_k": top_k,
        "include_metadata": True,
        "namespace": namespace,
    }
    if sparse_vector:
        kwargs["sparse_vector"] = sparse_vector
    return index.query(**kwargs)

### 3. LLM Service (Multi-Provider Support)

In [ ]:
import google.genai as genai
from google.genai import types

def get_gemini_client():
    return genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

async def generate_response(prompt, provider="gemini"):
    if provider == "gemini":
        client = get_gemini_client()
        resp = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=[types.Part.from_text(prompt)]
        )
        return resp.text
    return "Provider not supported in notebook yet."

### 4. Document Processing (PDF/Docx Parsing)

In [ ]:
from pypdf import PdfReader
from docx import Document

def process_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    if ext == ".pdf":
        reader = PdfReader(file_path)
        text = " ".join([p.extract_text() for p in reader.pages])
    elif ext == ".docx":
        doc = Document(file_path)
        text = " ".join([p.text for p in doc.paragraphs])
    return text